# NeMo GPU worker — Spanish STT/TTS (Acme Telco / Ava)

Runtime → **Change runtime type** → **T4 GPU**.

This notebook only runs GPU STT/TTS. Your PC keeps AudioSocket, VAD, LLM mocks, and metrics.

**Colab terms:** do not use JavaScript or hidden loops to defeat idle timeouts. Keep this tab
open during a demo session.


In [ ]:
%pip install -q "nemo_toolkit[asr,tts]" fastapi uvicorn soundfile python-multipart pyngrok
!nvidia-smi -L

In [ ]:
import os
import secrets
import sys
import threading
from pathlib import Path

REPO = Path("/content/realtime-voice-agentv2")
if not REPO.exists():
    !git clone --depth 1 https://github.com/haroldcastro-devai/realtime-voice-agentv2.git /content/realtime-voice-agentv2
sys.path.insert(0, str(REPO / "colab"))

token = secrets.token_urlsafe(32)
os.environ["INFERENCE_TOKEN"] = token
os.environ.setdefault("NEMO_TTS_SPEC", "tts_es_fastpitch_multispeaker")
os.environ.setdefault("NEMO_TTS_VOCODER", "tts_es_hifigan")

from nemo.collections.tts.models import FastPitchModel, HifiGanModel

spec_name = os.environ["NEMO_TTS_SPEC"]
vocoder_name = os.environ["NEMO_TTS_VOCODER"]
spec = FastPitchModel.from_pretrained(spec_name).eval().cuda()
vocoder = HifiGanModel.from_pretrained(vocoder_name).eval().cuda()
print("NeMo TTS loaded:", spec_name, "+", vocoder_name)

print("RVA_COLAB_TOKEN (Bearer):", token)

https_url = "https://YOUR-TUNNEL-HOST"
ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "")
if ngrok_token:
    from pyngrok import ngrok

    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8000, bind_tls=True)
    https_url = tunnel.public_url
print("RVA_COLAB_URL (HTTPS base):", https_url)


In [ ]:
import uvicorn
from nemo_server import app

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=_serve, daemon=True).start()
print("OpenAI STT:", https_url + "/v1/audio/transcriptions")
print("OpenAI TTS:", https_url + "/v1/audio/speech")
